# CNN-based Density Ratio Estimation (DRE) for Species Distribution Modelling

This notebook implements **four density ratio estimation methods** using a shared CNN encoder with partial convolutions for handling missing data in spatial patches.

## Methods Implemented
1. **Classifier DRE** — logistic regression on the density ratio  
2. **Deep KuLSIF** — direct least-squares ratio estimation via the KuLSIF objective  
3. **Deep KLIEP** — KL-divergence–based ratio estimation  
4. **Deep uLSIF** — unconstrained least-squares importance fitting  

## Architecture
All methods share the same backbone:
- **Encoder**: Hybrid partial-convolution CNN with channel-aware input, dilated convolutions, and masked global average pooling  
- **Head**: Small MLP producing either logits, log-ratios, or non-negative ratios depending on the method

## Evaluation
- **ROC-AUC**: discrimination between presence (p₁) and background (p₀)  
- **Continuous Boyce Index**: calibration of the predicted suitability surface

## 1. Imports

In [ ]:
import os
import random
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

## 2. Configuration

All hyperparameters are centralised here. Method-specific parameters are grouped at the end.

In [ ]:
# ── Paths ──────────────────────────────────────────────────
CV_DIR   = "data/cv_patches_13"
TEST_DIR = "data/test_patches_13"

CV_X_PATH    = os.path.join(CV_DIR, "X.npy")
CV_M_PATH    = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH    = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = "models/"   # override per method below

# ── Training ──────────────────────────────────────────────
BATCH_SIZE   = 256
NUM_WORKERS  = 0
MAX_EPOCHS   = 100
PATIENCE     = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-3
SEED         = 42

# ── Architecture ──────────────────────────────────────────
PATCH_SIZE  = 13
EMB_DIM     = 32
HIDDEN_DIMS = [32]
DROPOUT     = 0.35
CLIP_LOGITS = 10.0

# ── Data augmentation ────────────────────────────────────
USE_FLIPS = True
NOISE_STD = 0.04

# ── Model selection ──────────────────────────────────────
USE_AUC_FLOOR      = True
AUC_FLOOR           = 0.80
LOSS_WINDOW_DELTA   = 0.03

# ── Numeric stability ───────────────────────────────────
MAX_ABS_LOG_RATIO = 50.0

# ── Ensemble ─────────────────────────────────────────────
ENSEMBLE_IN_LOGSPACE = True

# ── Method-specific: Classifier DRE (BCE + Ranking) ─────
LAMBDA_RANK      = 0.05
MAX_RANK_PAIRS   = 4096
RANK_TEMPERATURE = 1.0
AUC_SOFTMAX_TAU  = 50.0   # for softmax(AUC) ensemble weights

# ── Method-specific: KuLSIF ─────────────────────────────
KU_LAMBDA_NORM  = 1e-4
KU_LAMBDA_MEAN1 = 1e-2
KU_RATIO_EPS    = 1e-8

# ── Method-specific: uLSIF ──────────────────────────────
RATIO_EPS = 1e-8
RATIO_MAX = 1e3

## 3. Device & Reproducibility

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

## 4. Evaluation Metrics

- **Continuous Boyce Index**: Spearman correlation between binned predicted-to-expected ratio and bin centre, following Hirzel et al. (2006).  
- **ROC-AUC**: standard discrimination metric.

In [ ]:
def _average_ranks(x: np.ndarray) -> np.ndarray:
    """Compute average ranks with tie handling."""
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x, y = np.asarray(x, float), np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    """Continuous Boyce Index (Hirzel et al. 2006)."""
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)

    s_bg, s_pr = s[y == 0], s[y == 1]
    if s_bg.size < min_per_group or s_pr.size < min_per_group:
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        nbk = int(((s_bg >= a) & (s_bg < b)).sum())
        if nbk == 0:
            continue
        npk = int(((s_pr >= a) & (s_pr < b)).sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y, s = np.asarray(y_true, int), np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y, s = np.asarray(y_true, int), np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )

## 5. Score Conversion Helpers

Functions to convert between logits, log-ratios, ratios, and bounded \[0,1\] scores.

In [ ]:
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.clip(np.asarray(x, np.float64), -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    """Classifier logit → log p1/p0 via prior correction."""
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits + np.log(pi0 / pi1)


def logits_to_bounded_score(logits: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    return sigmoid_np(logits_to_logratio(logits, pi0, pi1))


def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.clip(np.asarray(r, np.float64), 0.0, np.inf)
    return r / (1.0 + r)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    return sorted(int(v) for v in np.unique(fold_cv))

## 6. Dataset & Augmentation

Spatial patches are augmented with random flips and Gaussian noise (applied only to valid pixels via the mask).

In [ ]:
def augment_patch(values: torch.Tensor, mask: torch.Tensor):
    """Random flips + masked Gaussian noise.  values/mask: (C, H, W)."""
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask   = torch.flip(mask,   dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask   = torch.flip(mask,   dims=[2])
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise * (mask > 0).to(values.dtype)
    return values, mask


class PatchDREDataset(Dataset):
    """Dataset serving (values, mask, label) triples for patch-based DRE."""

    def __init__(self, X_values, X_masks, y, indices, train: bool):
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y  = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y  = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y

## 7. Encoder — Partial Convolution Blocks



In [ ]:
class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for raw input covariates.
    Input:  x, m both (B, C, H, W)
    Output: y (B, C_out, H', W'),  m_spatial (B, 1, H', W')
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_weight", torch.ones(in_channels, 1, kH, kW))
        self.stride, self.padding = stride, padding

    def forward(self, x, m):
        if m is None:
            m = torch.ones_like(x)
        m = (m > 0).to(dtype=x.dtype)
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))
        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale * (m_sum > 0).to(x.dtype)
        y = self.conv(x_norm)
        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(x.dtype)
        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """
    Spatial partial conv in feature space.
    Input:  x (B, C, H, W),  m (B, 1, H, W)
    Supports dilation for expanded receptive field without extra pooling.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                 dilation=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride, self.padding, self.dilation = stride, padding, dilation

    def forward(self, x, m):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        m = (m > 0).to(dtype=x.dtype)
        y = self.conv(x * m)
        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride,
                             padding=self.padding, dilation=self.dilation)
            m_out = (m_sum > 0).to(x.dtype)
        y = y * (self.kernel_area / (m_sum + self.eps)) * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First encoder block: channel-aware partial conv → BN → ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p  = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)
        return F.relu(self.bn(x), inplace=True), m


class CPConvBlock(nn.Module):
    """Later encoder blocks: spatial partial conv → BN → ReLU."""
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.p  = SpatialPartialConv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x, m):
        x, m = self.p(x, m)
        return F.relu(self.bn(x), inplace=True), m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x, m = self.pool(x), self.pool(m)
        return x, (m > 0).to(x.dtype)


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        m = (m > 0).to(x.dtype)
        return (x * m).sum(dim=(2, 3)) / m.sum(dim=(2, 3)).clamp_min(self.eps)

### Patch Encoder (13×13)




In [ ]:
class PatchEncoder13(nn.Module):
    """
    Hybrid partial-conv encoder for 13×13 patches.
    Includes a dilated (d=2) block after pooling to widen the
    receptive field without additional spatial reduction.
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.pool = MaskedMaxPool2d(2, 2)          # 13 → 6
        self.b3 = CPConvBlock(16, 32, dilation=1)
        self.b4 = CPConvBlock(32, 32, dilation=2)  # dilated
        self.b5 = CPConvBlock(32, 32, dilation=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m_cov = (x_mask > 0).to(x_val.dtype)
        x, m = self.b1(x_val, m_cov)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)
        return self.proj(self.gap(x, m))

## 8. MLP Heads



In [ ]:
class MLPLogitHead(nn.Module):
    """MLP producing an unconstrained scalar (logit or f-value)."""

    def __init__(self, in_dim, hidden_dims=(32,), dropout=0.2, clip=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)] if dropout > 0 \
                else [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip = clip

    def forward(self, x):
        out = self.net(x).squeeze(-1)
        if self.clip is not None:
            out = torch.clamp(out, -self.clip, self.clip)
        return out


class MLPSoftplusHead(nn.Module):
    """MLP producing a strictly positive ratio r(x) = softplus(g(x)) + eps."""

    def __init__(self, in_dim, hidden_dims=(32,), dropout=0.2, clip=None, eps=1e-8, r_max=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)] if dropout > 0 \
                else [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip = clip
        self.eps  = eps
        self.r_max = r_max

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip is not None:
            g = torch.clamp(g, -self.clip, self.clip)
        r = F.softplus(g) + self.eps
        if self.r_max is not None:
            r = torch.clamp(r, max=self.r_max)
        return r

## 9. Full Model Wrappers

In [ ]:
class CNNLogitDRE(nn.Module):
    """Encoder + logit head (for Classifier DRE and KLIEP)."""
    def __init__(self, in_ch, emb_dim=EMB_DIM, hidden_dims=HIDDEN_DIMS,
                 dropout=DROPOUT, clip=CLIP_LOGITS):
        super().__init__()
        self.encoder = PatchEncoder13(in_ch, emb_dim)
        self.head    = MLPLogitHead(emb_dim, hidden_dims, dropout, clip)

    def forward(self, x_val, x_mask):
        return self.head(self.encoder(x_val, x_mask))


class CNNRatioDRE(nn.Module):
    """Encoder + softplus head (for KuLSIF and uLSIF)."""
    def __init__(self, in_ch, emb_dim=EMB_DIM, hidden_dims=HIDDEN_DIMS,
                 dropout=DROPOUT, clip=CLIP_LOGITS, eps=1e-8, r_max=None):
        super().__init__()
        self.encoder = PatchEncoder13(in_ch, emb_dim)
        self.head    = MLPSoftplusHead(emb_dim, hidden_dims, dropout, clip, eps, r_max)

    def forward(self, x_val, x_mask):
        return self.head(self.encoder(x_val, x_mask))

## 10. Loss Functions

All four DRE objectives, self-contained and shareable.

In [ ]:
# ═══════════════════════════════════════════════════════════
#  1) Classifier DRE: BCE + pairwise ranking
# ═══════════════════════════════════════════════════════════

def pairwise_ranking_loss(logits, y, max_pairs=MAX_RANK_PAIRS, temperature=RANK_TEMPERATURE):
    """Pairwise logistic ranking: encourages logits(pos) > logits(neg)."""
    pos, neg = logits[y > 0.5], logits[y <= 0.5]
    if pos.numel() == 0 or neg.numel() == 0:
        return logits.new_tensor(0.0)
    diff = (pos[:, None] - neg[None, :]) / max(1e-8, temperature)
    if diff.numel() > max_pairs:
        idx = torch.randint(0, diff.numel(), (max_pairs,), device=logits.device)
        diff = diff.reshape(-1)[idx]
    return F.softplus(-diff).mean()


def classifier_dre_loss(logits, y, bce_fn):
    """BCE + λ · pairwise ranking loss."""
    return bce_fn(logits, y) + LAMBDA_RANK * pairwise_ranking_loss(logits, y)


# ═══════════════════════════════════════════════════════════
#  2) KuLSIF:  ½ E_p0[r²] − E_p1[r]  + regularisers
# ═══════════════════════════════════════════════════════════

def kulsif_loss(ratio, y, lambda_norm=KU_LAMBDA_NORM, lambda_mean1=KU_LAMBDA_MEAN1):
    """
    Direct least-squares ratio estimation (Kanamori et al., 2009).
    ratio: r(x) > 0   |   y=1 → p1 (numerator),  y=0 → p0 (denominator)
    """
    pos, neg = (y == 1), (y == 0)
    if pos.sum() == 0 or neg.sum() == 0:
        return ratio.mean() * 0.0
    r_neg = ratio[neg]
    loss = 0.5 * r_neg.pow(2).mean() - ratio[pos].mean()
    if lambda_norm > 0:
        loss = loss + lambda_norm * ratio.pow(2).mean()
    if lambda_mean1 > 0:
        loss = loss + lambda_mean1 * (r_neg.mean() - 1.0) ** 2
    return loss


# ═══════════════════════════════════════════════════════════
#  3) KLIEP:  −E_p1[f] + log E_p0[exp(f)]
# ═══════════════════════════════════════════════════════════

def _logmeanexp(t, dim=0):
    return torch.logsumexp(t, dim=dim) - torch.log(torch.tensor(t.size(dim), device=t.device, dtype=t.dtype))


def kliep_loss(f_pos, f_neg):
    """KL-importance estimation procedure (Sugiyama et al., 2008)."""
    f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return -(f_pos.mean() - _logmeanexp(f_neg, dim=0))


# ═══════════════════════════════════════════════════════════
#  4) uLSIF:  ½ E_p0[r²] − E_p1[r]  (no regulariser)
# ═══════════════════════════════════════════════════════════

def ulsif_loss(ratio, y):
    """Unconstrained least-squares importance fitting (Kanamori et al., 2009)."""
    r0 = ratio[y <= 0.5]
    r1 = ratio[y >  0.5]
    if r0.numel() == 0 or r1.numel() == 0:
        return torch.zeros((), device=ratio.device, dtype=ratio.dtype)
    return 0.5 * r0.pow(2).mean() - r1.mean()

## 11. Ensemble Weighting Schemes

In [ ]:
def softmax_auc_weights(fold_aucs: List[float], tau: float = AUC_SOFTMAX_TAU) -> np.ndarray:
    """w_k ∝ exp(τ · (auc_k − mean_auc)),  normalised."""
    a = np.asarray(fold_aucs, np.float64)
    if a.size == 0:
        return np.array([], np.float64)
    m = np.isfinite(a)
    if m.sum() == 0:
        return np.ones_like(a) / len(a)
    x = np.where(m, a - np.nanmean(a), -1e9) * tau
    ex = np.exp(np.clip(x - np.max(x[m]), -200, 200))
    ex[~m] = 0.0
    s = ex.sum()
    return ex / s if (s > 0 and np.isfinite(s)) else np.ones_like(a) / len(a)


def auc_linear_weights(fold_aucs: List[float], power=1.0, floor=0.5) -> np.ndarray:
    """w_k ∝ (auc_k − floor)^power,  normalised."""
    a = np.asarray(fold_aucs, np.float64)
    if a.size == 0 or not np.isfinite(a).any():
        return np.array([], np.float64)
    a = np.where(np.isfinite(a), a, np.nan)
    w = np.clip(a - floor, 0.0, None)
    if np.all(w == 0):
        m = np.isfinite(a)
        w = np.zeros_like(a)
        if m.sum() > 0:
            w[m] = 1.0 / m.sum()
        return w
    w = w ** power
    return w / w.sum()

## 12. Training — Standard Loop



In [ ]:
def _train_fold_standard(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
    model_factory,          # () -> nn.Module
    loss_fn,                # (model, xv, xm, yb) -> (loss, raw_output_np)
    output_to_score,        # (all_raw, fold_info) -> score01
    ckpt_prefix: str = "cnn_dre_fold",
    extra_state_fn=None,    # (fold_info) -> dict  (extra fields for checkpoint)
):
    """
    Train one CV fold.  Returns (best_val_metrics, val_score01, val_indices).
    """
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx   = np.where(f == fold_id)[0]

    pi1 = float((y_cv[train_idx] == 1).mean())
    pi0 = 1.0 - pi1
    fold_info = dict(pi0=pi0, pi1=pi1, fold_id=fold_id)

    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
          f"pi1={pi1:.6f}, pi0={pi0:.6f}")

    ds_tr  = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx,   train=False)
    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = model_factory().to(device)
    opt   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()
        total_loss, all_out, all_y = 0.0, [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(train):
                loss, raw = loss_fn(model, xv, xm, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()
            total_loss += loss.item() * yb.size(0)
            all_out.append(raw)
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_out = np.concatenate(all_out)
        all_y   = np.concatenate(all_y)
        score01 = output_to_score(all_out, fold_info)
        mets = compute_boyce_and_auc(all_y, score01)
        mets["loss"] = total_loss
        return mets, all_out, all_y

    candidates, best_loss_seen, no_improve = [], np.inf, 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _           = run_epoch(dl_tr, opt)
        va, val_raw, val_y = run_epoch(dl_val, None)

        print(f"[fold {fold_id}] epoch {ep:03d} | "
              f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
              f"val   loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}")

        auc_ok = (not USE_AUC_FLOOR) or (np.isfinite(va["ROC_AUC"]) and va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state":      {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": X_values.shape[1],
                "patch_size":        PATCH_SIZE,
                "emb_dim":           EMB_DIM,
                "hidden_dims":       HIDDEN_DIMS,
            }
            if extra_state_fn:
                state.update(extra_state_fn(fold_info))

            candidates.append(dict(
                loss=float(va["loss"]),
                boyce=float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                auc=float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                state=state, val_raw=val_raw, val_y=val_y, epoch=ep,
            ))
            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if not candidates:
        raise RuntimeError(f"[fold {fold_id}] No epoch met AUC floor={AUC_FLOOR}.")

    # Select: max Boyce within loss window
    loss_min = min(c["loss"] for c in candidates)
    window   = [c for c in candidates if c["loss"] <= loss_min + LOSS_WINDOW_DELTA]
    best     = max(window, key=lambda c: (c["boyce"] if np.isfinite(c["boyce"]) else -np.inf, -c["loss"]))

    path = os.path.join(model_dir, f"{ckpt_prefix}_{fold_id}.pt")
    torch.save(best["state"], path)
    print(f"[fold {fold_id}] selected epoch={best['epoch']} | "
          f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} → {path}")

    val_score01 = output_to_score(best["val_raw"], fold_info)
    return dict(Boyce=best["boyce"], ROC_AUC=best["auc"], loss=best["loss"]), val_score01, val_idx

## 13. Training — KLIEP (separate pos/neg loaders)

KLIEP requires iterating over presence and background samples independently, so it uses a dedicated training loop.

In [ ]:
@torch.no_grad()
def _estimate_logZ(model, dl_neg):
    """log E_{p0}[exp(f(x))]  estimated on the full train-negative set."""
    model.eval()
    vals = []
    for xv, xm, _ in dl_neg:
        f = model(xv.to(device), xm.to(device))
        vals.append(torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO).cpu())
    f_all = torch.cat(vals)
    return float(_logmeanexp(f_all, dim=0).item())


@torch.no_grad()
def _predict_f(model, X_values, X_masks, indices):
    model.eval()
    idx, outs = np.asarray(indices, np.int64), []
    for s in range(0, len(idx), BATCH_SIZE):
        b = idx[s:s + BATCH_SIZE]
        f = model(torch.from_numpy(X_values[b].astype(np.float32)).to(device),
                  torch.from_numpy(X_masks[b].astype(np.float32)).to(device))
        outs.append(torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO).cpu().numpy())
    return np.concatenate(outs)


def _train_fold_kliep(
    fold_id, X_values, X_masks, y_cv, fold_cv, model_dir,
    ckpt_prefix="cnn_dre_kliep_fold",
):
    os.makedirs(model_dir, exist_ok=True)
    f = fold_cv.astype(int)
    train_idx, val_idx = np.where(f != fold_id)[0], np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    print(f"\n[fold {fold_id}] train pos={len(train_pos)}, neg={len(train_neg)}, val N={len(val_idx)}")

    dl_tr_pos = DataLoader(PatchDREDataset(X_values, X_masks, y_cv, train_pos, True),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(PatchDREDataset(X_values, X_masks, y_cv, train_neg, True),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg_full = DataLoader(PatchDREDataset(X_values, X_masks, y_cv, train_neg, False),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]
    dl_val_pos = DataLoader(PatchDREDataset(X_values, X_masks, y_cv, val_pos, False),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    dl_val_neg = DataLoader(PatchDREDataset(X_values, X_masks, y_cv, val_neg, False),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    in_ch = X_values.shape[1]
    model = CNNLogitDRE(in_ch).to(device)
    opt   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_kliep_epoch(dl_p, dl_n, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()
        it_p, it_n = iter(dl_p), iter(dl_n)
        total, n = 0.0, 0
        for _ in range(max(len(dl_p), len(dl_n))):
            try: xvp, xmp, _ = next(it_p)
            except StopIteration: it_p = iter(dl_p); xvp, xmp, _ = next(it_p)
            try: xvn, xmn, _ = next(it_n)
            except StopIteration: it_n = iter(dl_n); xvn, xmn, _ = next(it_n)
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvn, xmn = xvn.to(device), xmn.to(device)
            if train: opt_obj.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(train):
                loss = kliep_loss(model(xvp, xmp), model(xvn, xmn))
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()
            total += loss.item() * xvp.size(0)
            n += xvp.size(0)
        return total / max(1, n)

    candidates, best_loss, no_improve = [], np.inf, 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_kliep_epoch(dl_tr_pos, dl_tr_neg, opt)
        logZ    = _estimate_logZ(model, dl_tr_neg_full)
        va_loss = run_kliep_epoch(dl_val_pos, dl_val_neg, None)

        f_val = _predict_f(model, X_values, X_masks, val_idx)
        lr_val = np.clip(f_val - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        score01 = sigmoid_np(lr_val)
        mets = compute_boyce_and_auc(y_cv[val_idx], score01)

        print(f"[fold {fold_id}] epoch {ep:03d} | train KLIEP {tr_loss:.4f} | "
              f"val KLIEP {va_loss:.4f}, AUC {mets['ROC_AUC']:.4f}, Boyce {mets['Boyce']:.4f}, logZ {logZ:.4f}")

        auc_ok = (not USE_AUC_FLOOR) or (np.isfinite(mets["ROC_AUC"]) and mets["ROC_AUC"] >= AUC_FLOOR)
        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_ch, "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM, "hidden_dims": HIDDEN_DIMS, "logZ": float(logZ),
            }
            candidates.append(dict(
                loss=float(va_loss),
                boyce=float(mets["Boyce"]) if np.isfinite(mets["Boyce"]) else np.nan,
                auc=float(mets["ROC_AUC"]) if np.isfinite(mets["ROC_AUC"]) else np.nan,
                state=state, val_logratio=lr_val.copy(), epoch=ep,
            ))
            if va_loss < best_loss - 1e-9:
                best_loss, no_improve = float(va_loss), 0
            else:
                no_improve += 1
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}"); break

    if not candidates:
        raise RuntimeError(f"[fold {fold_id}] No epoch met AUC floor.")

    loss_min = min(c["loss"] for c in candidates)
    window = [c for c in candidates if c["loss"] <= loss_min + LOSS_WINDOW_DELTA]
    best   = max(window, key=lambda c: (c["boyce"] if np.isfinite(c["boyce"]) else -np.inf, -c["loss"]))

    path = os.path.join(model_dir, f"{ckpt_prefix}_{fold_id}.pt")
    torch.save(best["state"], path)
    print(f"[fold {fold_id}] selected epoch={best['epoch']} | "
          f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} → {path}")

    return dict(Boyce=best["boyce"], ROC_AUC=best["auc"], loss=best["loss"]), sigmoid_np(best["val_logratio"]), val_idx

## 14. Ensemble Prediction



In [ ]:
def ensemble_predict(
    X_values, X_masks, y, indices,
    model_dir, fold_ids, weights,
    model_factory,          # (state) -> nn.Module (loaded)
    forward_to_logratio,    # (model, xv, xm, state) -> log_ratio tensor
    ckpt_prefix="cnn_dre_fold",
    ensemble_logspace=ENSEMBLE_IN_LOGSPACE,
):
    """
    Generic ensemble prediction returning (score01, log_ratio, y) arrays.
    """
    models, states = [], []
    for fid in fold_ids:
        path  = os.path.join(model_dir, f"{ckpt_prefix}_{fid}.pt")
        state = torch.load(path, map_location="cpu")
        m     = model_factory(state).to(device)
        m.load_state_dict(state["model_state"])
        m.eval()
        models.append(m)
        states.append(state)

    w = np.asarray(weights, np.float64)
    if not np.isfinite(w).all() or w.sum() <= 0:
        w = np.ones(len(fold_ids)) / len(fold_ids)
    else:
        w = w / w.sum()
    wt = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, np.int64)
    all_s, all_lr, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            b   = idx[start:start + BATCH_SIZE]
            xv  = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
            xm  = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
            yb  = torch.from_numpy(y[b].astype(np.float32)).to(device)

            lrs = []
            for k, m in enumerate(models):
                lr_k = forward_to_logratio(m, xv, xm, states[k])
                lr_k = torch.clamp(lr_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                lrs.append(lr_k)

            if ensemble_logspace:
                lr_ens = sum(wt[k] * lrs[k] for k in range(len(models)))
            else:
                r_sum  = sum(wt[k] * torch.exp(lrs[k]) for k in range(len(models)))
                lr_ens = torch.log(torch.clamp(r_sum, min=1e-30))

            lr_ens = torch.clamp(lr_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            all_s.append(torch.sigmoid(lr_ens).cpu().numpy())
            all_lr.append(lr_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_s), np.concatenate(all_lr), np.concatenate(all_y)

## 15. Load Data

In [ ]:
set_seed()

X_cv  = np.load(CV_X_PATH)
M_cv  = np.load(CV_M_PATH)
y_cv  = np.load(CV_Y_PATH).astype(np.float32)
fold_cv = np.load(CV_FOLD_PATH).astype(int)

X_test = np.load(TEST_X_PATH)
M_test = np.load(TEST_M_PATH)
y_test = np.load(TEST_Y_PATH).astype(np.float32)

# Replace non-finite values
for arr in [X_cv, M_cv, X_test, M_test]:
    bad = ~np.isfinite(arr)
    if bad.any():
        print(f"Warning: {int(bad.sum())} non-finite values set to 0.")
        arr[bad] = 0.0

fold_ids = make_cv_indices(fold_cv)
in_ch = X_cv.shape[1]

print(f"CV:   X {X_cv.shape}, y {y_cv.shape}, folds {fold_ids}")
print(f"Test: X {X_test.shape}, y {y_test.shape}")

---
## Method 1 — Classifier DRE 



In [ ]:
METHOD1_DIR = os.path.join(MODEL_DIR, "classifier_bce_rank")
set_seed()

bce = nn.BCEWithLogitsLoss()

def loss_fn_clf(model, xv, xm, yb):
    logits = model(xv, xm)
    loss = classifier_dre_loss(logits, yb, bce)
    return loss, logits.detach().cpu().numpy()

def score_fn_clf(raw, info):
    return logits_to_bounded_score(raw, info["pi0"], info["pi1"])

def extra_state_clf(info):
    return {"pi0": info["pi0"], "pi1": info["pi1"]}

oof_1 = np.full_like(y_cv, np.nan, dtype=float)
aucs_1, boyces_1 = [], []

for fid in fold_ids:
    mets, s01, vidx = _train_fold_standard(
        fid, X_cv, M_cv, y_cv, fold_cv, METHOD1_DIR,
        model_factory=lambda: CNNLogitDRE(in_ch),
        loss_fn=loss_fn_clf,
        output_to_score=score_fn_clf,
        ckpt_prefix="cnn_dre_fold",
        extra_state_fn=extra_state_clf,
    )
    oof_1[vidx] = s01
    aucs_1.append(mets["ROC_AUC"])
    boyces_1.append(mets["Boyce"])

w1 = softmax_auc_weights(aucs_1)
print("\n[Method 1] OOF:", compute_boyce_and_auc(y_cv, oof_1))
print("[Method 1] Weights:", w1)

def _m1_factory(state):
    return CNNLogitDRE(state["in_value_channels"], state.get("emb_dim", EMB_DIM),
                       tuple(state.get("hidden_dims", HIDDEN_DIMS)))

def _m1_lr(model, xv, xm, state):
    logits = model(xv, xm)
    return logits + np.log(state["pi0"] / state["pi1"])

s1, lr1, yt1 = ensemble_predict(X_test, M_test, y_test, np.arange(len(y_test)),
                                METHOD1_DIR, fold_ids, w1, _m1_factory, _m1_lr, "cnn_dre_fold")
print("[Method 1] Test:", compute_boyce_and_auc(yt1, s1))

---
## Method 2 — Deep KuLSIF



In [ ]:
METHOD2_DIR = os.path.join(MODEL_DIR, "kulsif")
set_seed()

def loss_fn_ku(model, xv, xm, yb):
    r = model(xv, xm)
    loss = kulsif_loss(r, yb)
    return loss, r.detach().cpu().numpy()

def score_fn_ku(raw, _info):
    return ratio_to_bounded_score(raw)

oof_2 = np.full_like(y_cv, np.nan, dtype=float)
aucs_2, boyces_2 = [], []

for fid in fold_ids:
    mets, s01, vidx = _train_fold_standard(
        fid, X_cv, M_cv, y_cv, fold_cv, METHOD2_DIR,
        model_factory=lambda: CNNRatioDRE(in_ch, eps=KU_RATIO_EPS),
        loss_fn=loss_fn_ku,
        output_to_score=score_fn_ku,
        ckpt_prefix="cnn_kulsif_fold",
    )
    oof_2[vidx] = s01
    aucs_2.append(mets["ROC_AUC"])
    boyces_2.append(mets["Boyce"])

w2 = auc_linear_weights(aucs_2)
print("\n[Method 2] OOF:", compute_boyce_and_auc(y_cv, oof_2))
print("[Method 2] Weights:", w2)

def _m2_factory(state):
    return CNNRatioDRE(state["in_value_channels"], state.get("emb_dim", EMB_DIM),
                       tuple(state.get("hidden_dims", HIDDEN_DIMS)), eps=KU_RATIO_EPS)

def _m2_lr(model, xv, xm, _state):
    r = torch.clamp(model(xv, xm), min=1e-30)
    return torch.log(r)

s2, lr2, yt2 = ensemble_predict(X_test, M_test, y_test, np.arange(len(y_test)),
                                METHOD2_DIR, fold_ids, w2, _m2_factory, _m2_lr,
                                "cnn_kulsif_fold", ensemble_logspace=False)
print("[Method 2] Test:", compute_boyce_and_auc(yt2, s2))

---
## Method 3 — Deep KLIEP



In [ ]:
METHOD3_DIR = os.path.join(MODEL_DIR, "kliep")
set_seed()

oof_3 = np.full_like(y_cv, np.nan, dtype=float)
aucs_3, boyces_3 = [], []

for fid in fold_ids:
    mets, s01, vidx = _train_fold_kliep(
        fid, X_cv, M_cv, y_cv, fold_cv, METHOD3_DIR
    )
    oof_3[vidx] = s01
    aucs_3.append(mets["ROC_AUC"])
    boyces_3.append(mets["Boyce"])

w3 = auc_linear_weights(aucs_3)
print("\n[Method 3] OOF:", compute_boyce_and_auc(y_cv, oof_3))
print("[Method 3] Weights:", w3)

def _m3_factory(state):
    return CNNLogitDRE(state["in_value_channels"], state.get("emb_dim", EMB_DIM),
                       tuple(state.get("hidden_dims", HIDDEN_DIMS)))

def _m3_lr(model, xv, xm, state):
    f = torch.clamp(model(xv, xm), -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return f - state["logZ"]

s3, lr3, yt3 = ensemble_predict(X_test, M_test, y_test, np.arange(len(y_test)),
                                METHOD3_DIR, fold_ids, w3, _m3_factory, _m3_lr,
                                "cnn_dre_kliep_fold")
print("[Method 3] Test:", compute_boyce_and_auc(yt3, s3))

---
## Method 4 — Deep uLSIF



In [ ]:
METHOD4_DIR = os.path.join(MODEL_DIR, "ulsif")
set_seed()

def loss_fn_ulsif(model, xv, xm, yb):
    r = model(xv, xm)
    loss = ulsif_loss(r, yb)
    return loss, r.detach().cpu().numpy()

def score_fn_ulsif(raw, _info):
    return ratio_to_bounded_score(raw)

oof_4 = np.full_like(y_cv, np.nan, dtype=float)
aucs_4, boyces_4 = [], []

for fid in fold_ids:
    mets, s01, vidx = _train_fold_standard(
        fid, X_cv, M_cv, y_cv, fold_cv, METHOD4_DIR,
        model_factory=lambda: CNNRatioDRE(in_ch, eps=RATIO_EPS, r_max=RATIO_MAX),
        loss_fn=loss_fn_ulsif,
        output_to_score=score_fn_ulsif,
        ckpt_prefix="cnn_ulsif_fold",
    )
    oof_4[vidx] = s01
    aucs_4.append(mets["ROC_AUC"])
    boyces_4.append(mets["Boyce"])

w4 = auc_linear_weights(aucs_4)
print("\n[Method 4] OOF:", compute_boyce_and_auc(y_cv, oof_4))
print("[Method 4] Weights:", w4)

def _m4_factory(state):
    return CNNRatioDRE(state["in_value_channels"], state.get("emb_dim", EMB_DIM),
                       tuple(state.get("hidden_dims", HIDDEN_DIMS)), eps=RATIO_EPS,
                       r_max=state.get("ratio_max", RATIO_MAX))

def _m4_lr(model, xv, xm, _state):
    r = torch.clamp(model(xv, xm), min=1e-30)
    return torch.log(r)

s4, lr4, yt4 = ensemble_predict(X_test, M_test, y_test, np.arange(len(y_test)),
                                METHOD4_DIR, fold_ids, w4, _m4_factory, _m4_lr,
                                "cnn_ulsif_fold")
print("[Method 4] Test:", compute_boyce_and_auc(yt4, s4))

---
## Summary

In [ ]:
print("=" * 60)
print(f"{'Method':<25} {'CV AUC':>10} {'CV Boyce':>10} {'Test AUC':>10} {'Test Boyce':>10}")
print("-" * 60)

cv1 = compute_boyce_and_auc(y_cv, oof_1)
cv2 = compute_boyce_and_auc(y_cv, oof_2)
cv3 = compute_boyce_and_auc(y_cv, oof_3)
cv4 = compute_boyce_and_auc(y_cv, oof_4)

t1 = compute_boyce_and_auc(yt1, s1)
t2 = compute_boyce_and_auc(yt2, s2)
t3 = compute_boyce_and_auc(yt3, s3)
t4 = compute_boyce_and_auc(yt4, s4)

for name, cv, t in [
    ("BCE + Ranking", cv1, t1),
    ("KuLSIF",        cv2, t2),
    ("KLIEP",         cv3, t3),
    ("uLSIF",         cv4, t4),
]:
    print(f"{name:<25} {cv['ROC_AUC']:>10.4f} {cv['Boyce']:>10.4f} {t['ROC_AUC']:>10.4f} {t['Boyce']:>10.4f}")
print("=" * 60)